# Task 3: Multi-Agent Financial Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gobishangar1/CDAZZDEV-MLE-GobishangarVijayendran/blob/main/task3_agentic/notebooks/task3_agentic_research.ipynb)

**Candidate:** Gobishangar Vijayendran · **Ticker:** NVDA · **Framework:** LangGraph · **LLM:** Groq `openai/gpt-oss-120b`
(fallback `openai/gpt-oss-20b`)

The query (from the brief):
> Analyse the current financial health and market sentiment of NVDA. Identify the top three risks to its share
> price over the next 90 days and suggest one data-driven hedge strategy.

| Rubric part | Section |
|---|---|
| 3A All five tools implemented (types) | 2 |
| 3A Autonomous tool selection, observe-and-replan cycle, final report | 3 |
| 3A Error handling (tool failure, agent replans) | 4 |
| 3C Short-term memory (follow-up with no tool call) | 5 |
| 3B Roles and tool restriction | 6 |
| 3B Structured handoff, message trace, critique loop, end-to-end | 7 |
| 3C Persistent memory (JSON cache by ticker and date) | 8 |
| 3C Observability (`logs/agent_trace.jsonl`) + bonus dashboard | 9 |

All logic is in the `research_agents` package. This notebook only calls it and shows the results.

## How it works

```
3A  one agent, all five tools (LangGraph ReAct loop)

    START -> agent --(tool call?)--> tools -> agent -> ... --(no tool call)--> END -> report writer (JSON, Pydantic)
             ^ the LLM decides the next tool from what it has seen; Python only runs the tool and returns the result

3B  two agents, restricted tools, typed handoffs, one critique round

    Agent A (Data Analyst)    get_price_data, calculate_volatility, llm_sentiment   (no news, no web)
    Agent B (Research Writer) web_search, get_news                                  (no price tools)

    analyst --DataBrief--> writer_research -> writer_critique --ClarificationRequest--> analyst_clarify
            --ClarificationResponse--> writer_report -> ResearchReport (must say how the answer was used)
```

Why they need each other: Agent A can score sentiment but cannot fetch news. Agent B has the news but cannot score it,
and has no price or volatility data. So B sends its headlines to A in the clarification request.

## 0. Setup

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Task 3 notebook cells in rubric order', Date: 2026-10-11
import os, sys

REPO = "CDAZZDEV-MLE-GobishangarVijayendran"
if "google.colab" in sys.modules:                        # Colab: clone the repo and install
    if not os.path.exists(f"/content/{REPO}"):
        !git clone -q https://github.com/Gobishangar1/{REPO}.git /content/{REPO}
    %cd /content/{REPO}/task3_agentic
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":        # local Jupyter started inside notebooks/
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
# The Groq key comes from Colab Secrets (key icon on the left) or the environment. It is never printed.
from datetime import datetime

import pandas as pd
from IPython.display import Markdown, display

from research_agents.config import (DEFAULT_TICKER, RESEARCH_QUERY, TRACE_FILE, load_colab_secret, require_groq_key,
                                    setup_logging)

loaded = load_colab_secret("GROQ_API_KEY")
require_groq_key()
print("GROQ_API_KEY available: True | loaded from Colab Secrets:", loaded)
setup_logging()
pd.set_option("display.max_colwidth", 140)

TICKER = DEFAULT_TICKER
QUERY = RESEARCH_QUERY.format(ticker=TICKER)
STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")        # run ids in the trace file
print("Query:", QUERY)

In [ ]:
from research_agents.llm import make_agent_llm, make_json_llm, make_sentiment_llm
from research_agents.tools import ALL_TOOLS, ToolBox
from research_agents.trace import Tracer

agent_llm = make_agent_llm()          # tool calling: gpt-oss-120b, fallback gpt-oss-20b
json_llm = make_json_llm()            # structured outputs (report, clarification), JSON mode
sentiment_llm = make_sentiment_llm()  # the llm_sentiment tool: gpt-oss-20b, JSON mode
print("Models ready. Trace file:", TRACE_FILE)

## 1. Configuration
Every limit is a named constant (step budget, output caps, windows). Tool outputs sent to the LLM are capped to keep
the free-tier token budget (about 8K tokens per minute per model).

In [ ]:
import research_agents.config as cfg
names = ["AGENT_MODEL", "FALLBACK_MODEL", "SENTIMENT_MODEL", "AGENT_MAX_STEPS", "TOOL_OUTPUT_TO_LLM_CHARS",
         "TRACE_OUTPUT_CHARS", "DEFAULT_VOL_WINDOW", "TRADING_DAYS_PER_YEAR", "MAX_SENTIMENT_HEADLINES",
         "CLARIFICATION_ROUNDS", "JSON_ATTEMPTS"]
pd.DataFrame([(n, getattr(cfg, n)) for n in names], columns=["constant", "value"])

## 2. Task 3A: the five tools
Each tool has a Pydantic argument schema (what the LLM sees) and returns a Pydantic model. Here each tool is run
through `ToolBox.execute()`, the same path the agents use, so every call is also written to `agent_trace.jsonl`.

In [ ]:
tracer_tools = Tracer(run_id=f"tools-{STAMP}")
box = ToolBox(sentiment_llm=sentiment_llm, tracer=tracer_tools)

print("Tool schemas the LLM sees:")
for t in box.langchain_tools(list(ALL_TOOLS)):
    print(f"  {t.name}({', '.join(t.args)})")

results, rows = {}, []
calls = [("get_price_data", {"ticker": TICKER, "period": "1y"}),
         ("get_news", {"ticker": TICKER, "n": 10}),
         ("calculate_volatility", {"ticker": TICKER, "window": 30}),
         ("web_search", {"query": f"{TICKER} stock analyst outlook risks", "max_results": 5})]
for name, args in calls:
    text, status, result = box.execute("notebook", set(ALL_TOOLS), name, args)
    results[name] = result
    rows.append({"tool": name, "args": args, "status": status, "returns": type(result).__name__, "preview": text[:150]})

headlines = [h.title for h in results["get_news"].headlines] if results["get_news"] else ["NVIDIA shares rise"]
text, status, result = box.execute("notebook", set(ALL_TOOLS), "llm_sentiment", {"headlines": headlines[:10]})
results["llm_sentiment"] = result
rows.append({"tool": "llm_sentiment", "args": {"headlines": f"{len(headlines[:10])} headlines"}, "status": status,
             "returns": type(result).__name__, "preview": text[:150]})
pd.DataFrame(rows)

In [ ]:
p, v = results["get_price_data"], results["calculate_volatility"]
if p:
    print(f"Price: last close {p.last_close} | {p.period} return {p.period_return_pct}% | "
          f"{p.pct_from_52w_high}% from 52-week high\nTrend (computed by code): {p.trend}")
    display(pd.DataFrame([p.indicators]))
if v:
    print("\nVolatility:", v.interpretation)

In [ ]:
if results["get_news"]:
    display(pd.DataFrame([h.model_dump() for h in results["get_news"].headlines])[["title", "publisher", "published_at", "source"]])
if results["llm_sentiment"]:
    s = results["llm_sentiment"]
    print(f"Sentiment score {s.score:+.3f} ({s.label}), counts {s.counts}")
    display(pd.DataFrame([i.model_dump() for i in s.items]))
if results["web_search"]:
    display(pd.DataFrame([h.model_dump() for h in results["web_search"].results]))

## 3. Task 3A: one research agent, all five tools
The agent gets only the query. Nothing in the code fixes the order of tools: each turn, the LLM reads the results
so far and decides the next tool and its arguments (or decides it is done). The trace below shows the cycle:
`thinks` (the model's reasoning) -> `calls` (tool and arguments) -> `<-` (the observed result) -> next decision.

In [ ]:
from research_agents.report import report_to_markdown
from research_agents.single_agent import ResearchSession

tracer_3a = Tracer(run_id=f"3A-{STAMP}")
session = ResearchSession(TICKER, agent_llm, json_llm, ToolBox(sentiment_llm=sentiment_llm, tracer=tracer_3a),
                          tracer_3a)
result = session.research()

### 3.1 The decision path (observe -> decide -> act)
Each row is one decision by the LLM. The `reasoning` column is what it wrote before choosing the tool.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

def decision_table(messages):
    rows, step = [], 0
    status = {m.tool_call_id: (m.artifact or {}).get("status") for m in messages if isinstance(m, ToolMessage)}
    for m in messages:
        if isinstance(m, AIMessage):
            step += 1
            thought = (m.content or m.additional_kwargs.get("reasoning_content", "") or "").strip()
            for c in m.tool_calls or [{"name": "(final answer)", "args": {}, "id": None}]:
                rows.append({"step": step, "reasoning": " ".join(thought.split())[:220], "tool": c["name"],
                             "args": c["args"], "result": status.get(c["id"], "")})
    return pd.DataFrame(rows)

decision_table(result.run.messages)

### 3.2 Final report (three sections, validated with Pydantic)

In [ ]:
if result.report:
    display(Markdown(report_to_markdown(result.report)))
else:
    print("Report failed:", result.error)

## 4. Task 3A: error handling
### 4.1 Tool level: errors are returned to the agent as data, never raised
Each error result says what went wrong and what to try instead (only tools that this agent has).

In [ ]:
import json
checks = [("get_price_data", {"ticker": "NOT A TICKER"}),
          ("calculate_volatility", {"ticker": TICKER, "window": 2}),
          ("llm_sentiment", {"headlines": []}),
          ("no_such_tool", {})]
for name, args in checks:
    text, status, _ = box.execute("notebook", set(ALL_TOOLS), name, args)
    print(f"{name}({args}) -> {status}: {text[:230]}")

### 4.2 Agent level: web search is broken, the agent must find another way
`web_search` is replaced by a function that always fails (like a DuckDuckGo rate limit). The task asks the agent to
start with web search, so the failure happens first. The agent reads the error and its suggestion, and replans.

In [ ]:
from research_agents.agent import ReActAgent
from research_agents.prompts import RESEARCH_AGENT_SYSTEM

def broken_search(query, max_results):
    raise RuntimeError("202 Ratelimit (simulated DuckDuckGo outage)")

tracer_fail = Tracer(run_id=f"3A-failure-{STAMP}")
failing_agent = ReActAgent("Research Agent", list(ALL_TOOLS), RESEARCH_AGENT_SYSTEM.format(tools=", ".join(ALL_TOOLS)),
                           agent_llm, ToolBox(sentiment_llm=sentiment_llm, tracer=tracer_fail,
                                              search_backend=broken_search), tracer_fail, max_steps=5)
fail_run = failing_agent.run(f"Start with web_search: what are analysts and the news saying about the main risks "
                             f"to {TICKER} this quarter? Give a short answer with sources.")

In [ ]:
records = pd.DataFrame([{"tool": r.tool, "args": r.args, "status": r.status} for r in fail_run.records])
display(records)
failed = [i for i, r in enumerate(fail_run.records) if r.status in ("error", "empty")]
if failed:
    after = [r.tool for r in fail_run.records[failed[0] + 1:]]
    print(f"Failure at call {failed[0] + 1} ({fail_run.records[failed[0]].tool}). The agent continued with: {after}")
print("\nFinal answer:\n" + fail_run.final_text[:1200])

## 5. Task 3C: short-term memory
The follow-up runs in the **same session** (LangGraph checkpointer, same thread id). The earlier tool results are in
the conversation, so the agent answers from them. The check below counts tool calls made for the follow-up.

In [ ]:
thread = {"configurable": {"thread_id": session.thread_id}}
memory = session.agent.graph.get_state(thread).values["messages"]
print(f"Session memory: {len(memory)} messages, tool results: {[m.name for m in memory if isinstance(m, ToolMessage)]}")

follow = session.ask("Follow-up: what annualised volatility did you calculate, over which window, and what does it "
                     "mean for the hedge you suggested? Use the results you already have.")
print("\nAnswer:", follow.answer)
print("\nTool calls made for the follow-up:", follow.tool_calls)

## 6. Task 3B: two agents with separate tool access
Each agent is bound only to its own tools (the LLM never sees the others), and `ToolBox.execute()` also refuses
any tool outside the agent's list. Below: what happens if Agent B tries a price tool.

In [ ]:
from research_agents.multi_agent import AGENT_A, AGENT_B, TOOLS_A, TOOLS_B, run_research

display(pd.DataFrame([
    {"agent": AGENT_A, "role": "quantitative analysis", "tools": TOOLS_A, "output": "DataBrief (Pydantic)"},
    {"agent": AGENT_B, "role": "qualitative synthesis", "tools": TOOLS_B, "output": "ResearchReport (Pydantic)"}]))

guard = ToolBox(tracer=Tracer(run_id=f"3B-restriction-{STAMP}"))
for agent, tools, tool, args in [(AGENT_B, TOOLS_B, "get_price_data", {"ticker": TICKER}),
                                 (AGENT_A, TOOLS_A, "web_search", {"query": "NVDA news"})]:
    text, status, _ = guard.execute(agent, set(tools), tool, args)
    print(f"{agent} -> {tool}: {status.upper()} {text}")

## 7. Task 3B: the full pipeline (no manual step between query and report)
The trace prints what each agent said, every tool call and result, and every handoff with its schema:
`DataBrief` (A -> B), then the critique loop: `ClarificationRequest` (B -> A) and `ClarificationResponse` (A -> B),
then B's final report. `use_cache=False` forces a fresh run here; section 8 shows the cache.

In [ ]:
tracer_3b = Tracer(run_id=f"3B-{STAMP}")
record = run_research(TICKER, llm=agent_llm, json_llm=json_llm, sentiment_llm=sentiment_llm, tracer=tracer_3b,
                      use_cache=False)

### 7.1 Handoffs (typed messages between the agents)

In [ ]:
handoffs = [e for e in tracer_3b.events if e["kind"] == "handoff"]
display(pd.DataFrame([{"from": e["sender"], "to": e["receiver"], "schema": e["schema"],
                       "fields": ", ".join(e["payload"])} for e in handoffs]))
from research_agents.schemas import ClarificationRequest, ClarificationResponse, DataBrief
for e, schema in zip(handoffs, (DataBrief, ClarificationRequest, ClarificationResponse)):
    schema(**e["payload"])                              # each payload validates against its schema
print("All handoff payloads validate against their Pydantic schemas.")

### 7.2 Critique loop: request, response, and how the report used it

In [ ]:
req, resp = record["clarification"]["request"], record["clarification"]["response"]
print("Agent B asked:", req["question"])
print("Requested data:", req["requested_data"])
print(f"Headlines sent to Agent A: {len(req['headlines'])}")
print("Reason:", req["reason"])
print("\nAgent A answered:", resp["answer"])
print("Agent A used tools:", resp["tools_used"])
print(json.dumps(resp["data"], indent=1)[:1500])
if record["report"]:
    print("\nHow Agent B incorporated it:", record["report"]["clarification_used"])

### 7.3 Final report from Agent B

In [ ]:
if record["report"]:
    display(Markdown(report_to_markdown(record["report"])))
else:
    print("Report failed:", record["error"])
print("\nTrace summary:", record["trace_summary"])

## 8. Task 3C: persistent memory
The brief was saved as `memory/<TICKER>_<date>.json`. A second run with the same ticker detects the file and loads it:
no agent and no tool runs (0 tool calls in this run's trace).

In [ ]:
tracer_cache = Tracer(run_id=f"3C-cache-{STAMP}")
again = run_research(TICKER, llm=agent_llm, json_llm=json_llm, sentiment_llm=sentiment_llm, tracer=tracer_cache)
print("from_cache:", again["from_cache"], "| file:", again["path"])
print("tool calls in this run:", len(tracer_cache.tool_calls()))
print("same report as the first run:", again["report"] == record["report"])
from pathlib import Path
print("\nmemory/ files:", sorted(p.name for p in Path("memory").glob("*.json")))
print("keys in the saved brief:", list(again))

## 9. Task 3C: observability
Every tool call is one line in `logs/agent_trace.jsonl`: `ts, run_id, agent, tool, inputs, output` (truncated to
200 characters), `duration_ms` (wall clock) and `status`. Agent messages, handoffs and cache events are logged too.

In [ ]:
from research_agents.trace_view import load_events, runs, tool_calls

events = load_events(TRACE_FILE)
mine = events[events["run_id"].str.endswith(STAMP)]
print(f"{TRACE_FILE.name}: {len(events)} lines in total, {len(mine)} from this notebook run")
display(runs(mine))
calls_3b = tool_calls(events, f"3B-{STAMP}")
display(calls_3b[["start", "agent", "tool", "inputs", "status", "duration_ms", "output"]])
print("Longest logged output:", int(tool_calls(mine)["output"].str.len().max()), "characters (limit 200)")

In [ ]:
with open(TRACE_FILE, encoding="utf-8") as fh:
    last_call = [line for line in fh if '"kind": "tool_call"' in line][-1]
print(json.dumps(json.loads(last_call), indent=2))

**Bonus dashboard.** `dashboard/app.py` (Streamlit) reads the same file and shows each run: totals, a timeline of
tool calls per agent coloured by status, time per tool, the tool-call table, agent messages and handoffs.
Run it locally from `task3_agentic/`: `pip install -r requirements-dashboard.txt`, then
`streamlit run dashboard/app.py`.

## 10. Tests (offline: fake Yahoo, fake search, scripted models)

In [ ]:
!python -m pytest -rs

## 11. Save the outputs
The reports go to `outputs/`. In Colab, a zip with the trace, the memory file and the reports is downloaded, so they
can be committed to the repository (`task3_agentic/logs/agent_trace.jsonl` is a required deliverable).

In [ ]:
import zipfile
from research_agents.config import PACKAGE_ROOT

out = PACKAGE_ROOT / "outputs"
out.mkdir(exist_ok=True)
if result.report:
    (out / f"{TICKER}_3A_single_agent_report.md").write_text(report_to_markdown(result.report), encoding="utf-8")
if record["report"]:
    (out / f"{TICKER}_3B_multi_agent_report.md").write_text(report_to_markdown(record["report"]), encoding="utf-8")

files = [TRACE_FILE, *sorted((PACKAGE_ROOT / "memory").glob("*.json")), *sorted(out.glob("*.md"))]
with zipfile.ZipFile("task3_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f.relative_to(PACKAGE_ROOT))
        print("added", f.relative_to(PACKAGE_ROOT))
if "google.colab" in sys.modules:
    from google.colab import files as colab_files
    colab_files.download("task3_outputs.zip")

## Appendix: all prompts
Prompts are constants in `research_agents/prompts.py` (system = rules, human = data). They are printed here for
transparency, as the citation policy asks.

In [ ]:
from research_agents.prompts import all_prompts
for name, text in all_prompts().items():
    print(f"===== {name} =====\n{text}\n")